# 02 — Covariate shift (when P(X) moves)

**Covariate shift** means the input distribution changes while the conditional label rule P(Y|X) stays fixed. LedgerRoute simulates marketing pushing spend from offline procurement to online checkout (`channel_online` share rises from ~25% toward ~80%).

Symptoms in production:
- Global accuracy may look flat early on.
- Accuracy on the **growing segment** degrades first.
- PSI on one feature can stay low while the **mix plot** already tells the story.


In [ ]:
# From repo root: pip install -e ".[dev]"
%matplotlib inline

import pandas as pd

from drift_lab import StreamConfig, build_ledger_route_model, generate_stream
from drift_lab.analysis import accuracy_by_group, attach_predictions, compare_windows
from drift_lab.detectors import windowed_psi_series
from drift_lab.viz import channel_mix_figure, psi_series_figure

cfg = StreamConfig()
ref = generate_stream("stable", cfg)
cov = generate_stream("covariate_gradual", cfg)
model = build_ledger_route_model(cfg)  # trained on stable days 0–29

ref_win = ref[ref["day"] < 30]
cov_scored = attach_predictions(model, cov)
print("Reference rows:", len(ref_win), "| Covariate stream rows:", len(cov))


## Visual: channel mix over time


In [ ]:
fig, mix_meta = channel_mix_figure(cov)
mix_meta


## PSI and KS: reference vs late traffic


In [ ]:
late = cov[cov["day"] >= 90]
rows = []
for feat in ["channel_online", "log_amount", "foreign_flag"]:
    rows.append(compare_windows(ref_win, late, feat))
pd.DataFrame(rows).round(4)


## Segment accuracy (frozen model)


In [ ]:
acc = accuracy_by_group(cov_scored, "channel_online")
acc.assign(accuracy=acc["accuracy"].round(4))


## Windowed PSI on `log_amount` along the stream


In [ ]:
psi_df = windowed_psi_series(
    ref_win["log_amount"].values,
    cov["log_amount"].values,
    window=2000,
    step=400,
)
fig, psi_meta = psi_series_figure(psi_df)
psi_meta


## Takeaway

- Plot **mix and segments** before trusting a single PSI number.
- PSI below 0.1 is not a clean bill of health if conditional accuracy on the online tail is wrong.
- Response options: importance weighting, explicit channel feature, or scheduled retrain—after segment eval, not on PSI alone.
